# Lab 4.1: Context Hazard Linter

**Day 1 - Topic 4**

Goal: audit a repository's context files for stale paths, conflicting directives, unscoped rules, duplicate lines, and verbose tool output.

> Open this notebook in Google Colab:
> [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/context-claude-companion/blob/main/day1/lab4_1_context_hazard_linter/start/lab4_1_context_hazard_linter.ipynb)

In [ ]:
# No extra packages needed: this lab uses the Python standard library only.
# If running locally with the course venv already activated, you can skip this cell.

In [ ]:
import re
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/kpassoubady/context-claude-companion.git"


def find_repo_root() -> Path:
    """Use the local companion checkout if the notebook runs inside it; otherwise clone it (Colab)."""
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "day1" / "demos" / "fixtures").is_dir():
            return p
    clone = Path("context-claude-companion").resolve()
    if not clone.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
    return clone


REPO_ROOT = find_repo_root()
FIXTURES = REPO_ROOT / "day1" / "demos" / "fixtures"
print(f"Using fixtures in {FIXTURES}")


def check(label: str, ok: bool) -> None:
    """Print PASS when a result is correct, TODO while a step is unfinished."""
    print(("PASS  " if ok else "TODO  ") + label)

## Section 1: The repo under audit

`hazard_repo` has a root `CLAUDE.md`, one rule file, and a captured tool log. Read both instruction files and look for problems before the linter does.

In [ ]:
REPO = FIXTURES / "hazard_repo"
LOG_LINE_LIMIT = 500


def instruction_files() -> dict[str, str]:
    files = {"CLAUDE.md": (REPO / "CLAUDE.md").read_text()}
    for p in sorted((REPO / ".claude" / "rules").glob("*.md")):
        files[str(p.relative_to(REPO))] = p.read_text()
    return files


files = instruction_files()
for name, text in files.items():
    print(f"--- {name} ---")
    print(text)

## Section 2: Stale path references

A backticked path in an instruction file that does not exist wastes context and misleads the model.

In [ ]:
def detect_stale_paths(files: dict[str, str]) -> list[tuple[str, str]]:
    """Backticked paths in instructions that do not exist in the repo."""
    stale = []
    # TODO: For each (name, text) in files.items(), find backticked file paths with re.findall(r"`([\w./-]+\.\w+)`", text)
    # TODO: Keep a candidate only if it contains "/" and (REPO / candidate).exists() is False
    # TODO: Append (name, candidate) to stale for each candidate you keep
    return stale

In [ ]:
stale = detect_stale_paths(files)
print(stale)
# Expected: [('CLAUDE.md', 'docs/legacy_v1/architecture.md'), ('CLAUDE.md', 'scripts/old_deploy.sh')]
check("Two stale paths found in CLAUDE.md", [p for _, p in stale] == ["docs/legacy_v1/architecture.md", "scripts/old_deploy.sh"])

## Section 3: Conflicting directives

Provided: pair a tool that one file forbids with a line in another file that tells Claude to use it.

In [ ]:
FORBIDDEN = re.compile(r"never use\s+`?(\w+)`?|`(\w+)` is forbidden", re.I)
TOOL_ALIASES = {"unittest": ["unittest"], "requests": ["requests"]}


def detect_conflicts(files: dict[str, str]) -> list[str]:
    """A tool one file forbids that another file tells Claude to use."""
    conflicts = []
    for src, text in files.items():
        forbidden = {m for g in FORBIDDEN.findall(text) for m in g if m} & set(TOOL_ALIASES)
        for tool in forbidden:
            for other, other_text in files.items():
                if other == src:
                    continue
                for line in other_text.splitlines():
                    if re.search(rf"\b{tool}\b", line) and re.search(r"\b(use|run)\b", line, re.I):
                        conflicts.append(f"`{tool}` is forbidden in {src} but {other} says: {line.strip('- ').strip()}")
    return conflicts


conflicts = detect_conflicts(files)
for c in conflicts:
    print(c)

## Section 4: Unscoped rules and duplicate lines

A rule file without `paths:` frontmatter loads at every session start. A bullet repeated in the same file costs tokens twice.

In [ ]:
def detect_unscoped_rules(files: dict[str, str]) -> list[str]:
    """Rule files with no `paths:` frontmatter load on every session start."""
    # TODO: Return the names that start with ".claude/rules/" AND whose text, after lstrip(), does NOT start with "---"
    return []


def detect_duplicates(files: dict[str, str]) -> list[tuple[str, str]]:
    """Bullet lines that appear more than once in the same file."""
    dups = []
    # TODO: For each (name, text), keep a fresh set called seen and loop over text.splitlines()
    # TODO: Strip each line; for lines that start with "- ", append (name, line) to dups when line is already in seen
    # TODO: Add the line to seen after the check
    return dups

## Section 5: The full report

Run every check, plus a line count on the captured tool log. Change a fixture file and rerun to see the report change. Static checks do not replace human review or tests.

In [ ]:
print(f"=== Context Hazard Audit of fixtures/hazard_repo ({len(files)} instruction files) ===\n")

print(f"1. Stale path references ({len(stale)} found):")
for name, path in stale:
    print(f"   - {name}: `{path}` does not exist on disk")

print(f"\n2. Conflicting directives ({len(conflicts)} found):")
for c in conflicts:
    print(f"   - {c}")

unscoped = detect_unscoped_rules(files)
print(f"\n3. Rules without `paths:` frontmatter ({len(unscoped)} found):")
for name in unscoped:
    print(f"   - {name} loads on every session start")

dups = detect_duplicates(files)
print(f"\n4. Duplicate lines ({len(dups)} found):")
for name, line in dups:
    print(f"   - {name}: {line}")

log = REPO / "logs" / "last_tool_output.txt"
n = len(log.read_text().splitlines())
print("\n5. Verbose tool output:")
status = "over the limit; bound with `-q --tb=short` or path filters" if n > LOG_LINE_LIMIT else "within the limit"
print(f"   - {log.relative_to(REPO)} has {n:,d} lines ({status}, limit {LOG_LINE_LIMIT})")

# Expected: 2 stale paths, 2 conflicts, 1 unscoped rule, 1 duplicate line, and a 2,399-line log.
print()
check("Stale paths: 2 found", len(stale) == 2)
check("Conflicting directives: 2 found (requests, unittest)", len(conflicts) == 2)
check("Unscoped rules: .claude/rules/api-async.md", unscoped == [".claude/rules/api-async.md"])
check("Duplicate lines: 1 found in CLAUDE.md", len(dups) == 1 and dups[0][0] == "CLAUDE.md")
check("The tool log is over the 500-line limit", n > LOG_LINE_LIMIT)

In [ ]:
print("Lab complete.")
print("Takeaway: Static checks catch broken paths, unscoped rules, and explicit contradictions so human review and tests can focus on whether Claude actually follows the guidance.")